<a href="https://colab.research.google.com/github/JaimRM/QuantitativeFinance/blob/main/Vol_Corr_reciente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PREPARACIÓN DE DATOS**

In [15]:
import numpy as np
import pandas as pd
from scipy import stats

# ── Datos de rentabilidad mensual (%) extraídos de la imagen ──────────────────
mes_map = {"Ene":1,"Feb":2,"Mar":3,"Abr":4,"May":5,"Jun":6,
           "Jul":7,"Ago":8,"Sep":9,"Oct":10,"Nov":11,"Dic":12}

data = {
    "Año":  [2022, 2022, 2022, 2022, 2022, 2022, 2022, 2022, 2022,
             2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023,
             2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024,
             2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025,
             2026, 2026, 2026, 2026, 2026, 2026],
    "Mes":  ["Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun"],
    "Rentabilidad": [-2.78, 1.72, -9.17, 8.64, -1.40, 4.72, 7.37, 1.83, -8.13,
                     19.22, -7.16, -9.97, -5.17, -2.13, 6.88, 3.87, 4.85, -1.87, -4.14, 8.00, 2.64,
                     7.33, 1.28, 3.86, 0.00, 3.04, -12.66, 4.69, 0.37, 3.47, -2.49, -0.21, -1.63,
                     7.27, 4.52, 0.12, 3.60, 6.41, 3.97, 4.60, 7.00, 5.35, 1.10, 1.22, 5.35,
                     6.01, 5.08, -10.07, 6.27, 1.87, 2.18],
}

df = pd.DataFrame(data)

**RIESGO:**

**Volatilidad, Sharpe ratio, VaR, Expected Shortfall / CVaR**

In [16]:
df["Fecha"] = pd.to_datetime(
    df["Año"].astype(str) + "-" + df["Mes"].map(mes_map).astype(str) + "-01"
)
df = df.sort_values("Fecha").reset_index(drop=True)

# ── Función de análisis reutilizable ─────────────────────────────────────────
def analizar(nombre, r_arr):
    r = np.array(r_arr)
    n            = len(r)
    media_m      = np.mean(r)
    media_a      = media_m * 12
    vol_m        = np.std(r, ddof=1)
    vol_a        = vol_m * np.sqrt(12)
    sharpe       = media_a / vol_a if vol_a > 0 else np.nan

    # VaR histórico
    var95_h = np.percentile(r, 5)
    var99_h = np.percentile(r, 1)

    # VaR paramétrico (normal)
    var95_p = media_m - 1.645 * vol_m
    var99_p = media_m - 2.326 * vol_m

    # CVaR (Expected Shortfall) al 95%
    cvar95 = r[r <= var95_h].mean() if (r <= var95_h).any() else var95_h

    print(f"\n{'='*56}")
    print(f"  {nombre}")
    print(f"{'='*56}")
    print(f"  Meses analizados        : {n}")
    print(f"  Rentabilidad media mens.: {media_m:>7.2f} %")
    print(f"  Rentabilidad media anual: {media_a:>7.2f} %  (×12)")
    print(f"  Volatilidad mensual     : {vol_m:>7.2f} %")
    print(f"  Volatilidad anualizada  : {vol_a:>7.2f} %  (×√12)")
    print(f"  Sharpe aprox. (rf=0)    : {sharpe:>7.3f}")
    print(f"\n  ── VaR mensual ──────────────────────────────")
    print(f"  VaR 95% histórico       : {var95_h:>7.2f} %")
    print(f"  VaR 99% histórico       : {var99_h:>7.2f} %")
    print(f"  VaR 95% paramétrico     : {var95_p:>7.2f} %")
    print(f"  VaR 99% paramétrico     : {var99_p:>7.2f} %")
    print(f"  CVaR 95% (Exp. Shortf.) : {cvar95:>7.2f} %")

    return dict(vol_m=vol_m, vol_a=vol_a, sharpe=sharpe,
                var95_h=var95_h, var99_h=var99_h,
                var95_p=var95_p, var99_p=var99_p, cvar95=cvar95)

# ── 1. Período completo ───────────────────────────────────────────────────────
res_total = analizar("PERÍODO COMPLETO (abr-2022 → jun-2026)",
                     df["Rentabilidad"].values)

# ── 2. Últimos 2 años (desde jul-2024, primer mes completo tras jun-2024) ──────
cutoff = pd.Timestamp("2024-07-01")   # primer mes completo después del 18-jun-2024
df_2a  = df[df["Fecha"] >= cutoff].copy()
res_2a = analizar("ÚLTIMOS 2 AÑOS (jul-2024 → jun-2026)",
                  df_2a["Rentabilidad"].values)

# ── 3. Volatilidad por año (período completo) ─────────────────────────────────
print(f"\n{'='*56}")
print("  VOLATILIDAD ANUAL DESGLOSADA")
print(f"{'='*56}")
print(f"  {'Año':<6} {'Vol. mens.':>12} {'Vol. anual.':>13}")
print("  " + "-"*33)
for año, grp in df.groupby("Año"):
    vm = grp["Rentabilidad"].std(ddof=1)
    print(f"  {año:<6} {vm:>11.2f}% {vm*np.sqrt(12):>12.2f}%")

print(f"\n  Nota: VaR = pérdida máxima esperada a ese nivel de confianza")
print(f"  en un mes dado. CVaR = pérdida media cuando se supera el VaR. También conocido como Expected Shortfall")
print(f"{'='*56}\n")


  PERÍODO COMPLETO (abr-2022 → jun-2026)
  Meses analizados        : 51
  Rentabilidad media mens.:    1.70 %
  Rentabilidad media anual:   20.40 %  (×12)
  Volatilidad mensual     :    5.77 %
  Volatilidad anualizada  :   19.99 %  (×√12)
  Sharpe aprox. (rf=0)    :   1.021

  ── VaR mensual ──────────────────────────────
  VaR 95% histórico       :   -9.57 %
  VaR 99% histórico       :  -11.37 %
  VaR 95% paramétrico     :   -7.79 %
  VaR 99% paramétrico     :  -11.72 %
  CVaR 95% (Exp. Shortf.) :  -10.90 %

  ÚLTIMOS 2 AÑOS (jul-2024 → jun-2026)
  Meses analizados        : 24
  Rentabilidad media mens.:    2.75 %
  Rentabilidad media anual:   33.03 %  (×12)
  Volatilidad mensual     :    3.88 %
  Volatilidad anualizada  :   13.42 %  (×√12)
  Sharpe aprox. (rf=0)    :   2.460

  ── VaR mensual ──────────────────────────────
  VaR 95% histórico       :   -2.36 %
  VaR 99% histórico       :   -8.33 %
  VaR 95% paramétrico     :   -3.62 %
  VaR 99% paramétrico     :   -6.26 %
  CVaR 95%

**CORRELACIONES**

In [17]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

# ══════════════════════════════════════════════════════════════════════════════
# 1.  PORTFOLIO (jul-2024 → jun-2026)
# ══════════════════════════════════════════════════════════════════════════════
portfolio_raw = [
    (2024,7,4.69),(2024,8,0.37),(2024,9,3.47),(2024,10,-2.49),(2024,11,-0.21),(2024,12,-1.63),
    (2025,1,7.27),(2025,2,4.52),(2025,3,0.12),(2025,4,3.60),(2025,5,6.41),(2025,6,3.97),
    (2025,7,4.60),(2025,8,7.00),(2025,9,5.35),(2025,10,1.10),(2025,11,1.22),(2025,12,5.35),
    (2026,1,6.01),(2026,2,5.08),(2026,3,-10.07),(2026,4,6.27),(2026,5,1.87),(2026,6,2.18),
]
portfolio = pd.Series(
    {pd.Timestamp(y, m, 1): v for y, m, v in portfolio_raw},
    name="Portfolio"
)

# ══════════════════════════════════════════════════════════════════════════════
# 2.  BCOM
# ══════════════════════════════════════════════════════════════════════════════
bcom_raw = [
    (2024,7,-4.50),(2024,8,-0.38),(2024,9,4.43),(2024,10,-2.24),(2024,11,0.05),(2024,12,0.63),
    (2025,1,3.58),(2025,2,0.45),(2025,3,3.55),(2025,4,-5.14),(2025,5,-0.93),(2025,6,2.03),
    (2025,7,-0.82),(2025,8,1.58),(2025,9,1.79),(2025,10,2.56),(2025,11,2.90),(2025,12,-0.65),
    (2026,1,10.04),(2026,2,0.81),(2026,3,11.15),(2026,4,3.89),(2026,5,-3.84),(2026,6,-6.06),
]
bcom = pd.Series(
    {pd.Timestamp(y, m, 1): v for y, m, v in bcom_raw},
    name="BCOM"
)

# ══════════════════════════════════════════════════════════════════════════════
# 3.  LEER CSVs  (Investing.com formato español, sin thousands en parse)
# ══════════════════════════════════════════════════════════════════════════════
def parse_investing_csv(path, name):
    df = pd.read_csv(path)  # sin thousands para que Fecha no se corrompa
    df.columns = df.columns.str.strip().str.replace('\ufeff','')
    # Fecha: "01.06.2026" → Timestamp
    df["Fecha"] = pd.to_datetime(df["Fecha"], format="%d.%m.%Y")
    # % var.: "4,37%" o "-0,74%" o "+3,19%"
    pct = (df["% var."].astype(str)
           .str.replace('%','', regex=False)
           .str.replace('+','', regex=False)
           .str.replace(',','.', regex=False)
           .str.strip())
    df["ret"] = pd.to_numeric(pct, errors="coerce")
    s = df.set_index("Fecha")["ret"].rename(name).sort_index()
    return s.loc["2024-07-01":"2026-06-30"]

msci_world = parse_investing_csv("/content/Datos históricos del MSCI World.csv",    "MSCI World")
msci_em    = parse_investing_csv("/content/Datos históricos EEM.csv",               "MSCI EM")
gold       = parse_investing_csv("/content/Datos históricos GLD.csv",               "Gold")
wti        = parse_investing_csv("/content/Datos históricos Petróleo crudo WTI.csv","WTI Oil")
reit       = parse_investing_csv("/content/Datos históricos del MSCI U.S. REIT.csv","MSCI REIT")
agg        = parse_investing_csv("/content/Datos históricos USAG.csv",              "US Agg Bond")

# ══════════════════════════════════════════════════════════════════════════════
# 4.  UNIR  Y  VERIFICAR
# ══════════════════════════════════════════════════════════════════════════════
df = pd.concat([portfolio, msci_world, msci_em, gold, wti, reit, agg, bcom], axis=1).sort_index()
print(f"Filas antes de dropna: {len(df)}")
print(df.isna().sum())
df = df.dropna()
print(f"\nMeses con datos completos: {len(df)}")
print(f"Período: {df.index[0].strftime('%b-%Y')} → {df.index[-1].strftime('%b-%Y')}\n")

# ══════════════════════════════════════════════════════════════════════════════
# 5.  CORRELACIÓN
# ══════════════════════════════════════════════════════════════════════════════
corr = df.corr(method="pearson")
port_corr = corr["Portfolio"].drop("Portfolio").sort_values(ascending=False)

print("═"*54)
print("  CORRELACIÓN CON EL PORTFOLIO (Pearson, mensual)")
print("═"*54)
for asset, c in port_corr.items():
    bar = "█" * int(abs(c) * 20)
    sign = "+" if c >= 0 else "-"
    print(f"  {asset:<16} {c:>+.3f}  {sign}{bar}")

# ══════════════════════════════════════════════════════════════════════════════
# 6.  ESTADÍSTICAS COMPARATIVAS
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*70}")
print("  ESTADÍSTICAS COMPARATIVAS (jul-2024 → jun-2026)")
print(f"{'═'*70}")
print(f"  {'Activo':<18} {'Ret.media%':>10} {'Vol.mens%':>10} {'Vol.anual%':>11} {'Sharpe':>8}")
print("  " + "-"*60)
for col in df.columns:
    r  = df[col]
    mu = r.mean(); vm = r.std(ddof=1); va = vm * np.sqrt(12)
    sh = (mu * 12) / va if va > 0 else np.nan
    print(f"  {col:<18} {mu:>10.2f} {vm:>10.2f} {va:>11.2f} {sh:>8.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# 7.  HEATMAP
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 7),
                         gridspec_kw={"width_ratios": [2.2, 1]})
fig.patch.set_facecolor("#0f1117")

# heatmap completo
ax1 = axes[0]
labels = df.columns.tolist()
mat = corr.values; n = len(labels)
im = ax1.imshow(mat, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax1.set_xticks(range(n)); ax1.set_xticklabels(labels, rotation=35, ha="right", fontsize=9, color="white")
ax1.set_yticks(range(n)); ax1.set_yticklabels(labels, fontsize=9, color="white")
ax1.set_facecolor("#0f1117"); ax1.tick_params(colors="white")
for spine in ax1.spines.values(): spine.set_visible(False)
for i in range(n):
    for j in range(n):
        v = mat[i, j]
        ax1.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                 color="black" if abs(v) > 0.5 else "white", fontweight="bold")
ax1.set_title("Matriz de Correlación Completa", color="white", fontsize=13, pad=12, fontweight="bold")
cbar = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
cbar.outline.set_edgecolor("white")

# barras correlación vs Portfolio
ax2 = axes[1]
assets = port_corr.index.tolist(); values = port_corr.values
colors = ["#4CAF50" if v >= 0 else "#F44336" for v in values]
bars = ax2.barh(assets, values, color=colors, edgecolor="none", height=0.6)
ax2.set_xlim(-1, 1); ax2.axvline(0, color="white", linewidth=0.8, linestyle="--", alpha=0.5)
ax2.set_facecolor("#0f1117"); ax2.tick_params(colors="white", labelsize=9)
for spine in ax2.spines.values(): spine.set_edgecolor("#333")
ax2.set_xlabel("Correlación de Pearson", color="white", fontsize=9)
ax2.set_title("Correlación vs Portfolio", color="white", fontsize=13, pad=12, fontweight="bold")
for bar, val in zip(bars, values):
    xpos = val + 0.03 if val >= 0 else val - 0.03
    ax2.text(xpos, bar.get_y() + bar.get_height()/2,
             f"{val:+.3f}", va="center", ha="left" if val >= 0 else "right",
             color="white", fontsize=9, fontweight="bold")

plt.tight_layout(pad=2)
plt.show()

Filas antes de dropna: 24
Portfolio      0
MSCI World     0
MSCI EM        0
Gold           0
WTI Oil        0
MSCI REIT      0
US Agg Bond    0
BCOM           0
dtype: int64

Meses con datos completos: 24
Período: Jul-2024 → Jun-2026

══════════════════════════════════════════════════════
  CORRELACIÓN CON EL PORTFOLIO (Pearson, mensual)
══════════════════════════════════════════════════════
  MSCI EM          +0.729  +██████████████
  MSCI World       +0.675  +█████████████
  MSCI REIT        +0.553  +███████████
  Gold             +0.526  +██████████
  US Agg Bond      +0.479  +█████████
  BCOM             -0.223  -████
  WTI Oil          -0.529  -██████████

══════════════════════════════════════════════════════════════════════
  ESTADÍSTICAS COMPARATIVAS (jul-2024 → jun-2026)
══════════════════════════════════════════════════════════════════════
  Activo             Ret.media%  Vol.mens%  Vol.anual%   Sharpe
  ------------------------------------------------------------
  Portfoli